# 01 Exploratory Data Analysis (EDA) - MPLADS Scheme Analytics
**Smart India Hackathon 2026 — Problem Statement 26102**
Development of an AI-Powered System to Detect Anomalies, Fraud, and Inefficiencies in MPLAD Scheme Implementation.

This notebook conducts comprehensive exploratory analysis on the curated dataset of 98,632 MPLADS public works records compiled from official Lok Sabha and Rajya Sabha publications.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Style configurations
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Load cleaned dataset
data_path = Path("../data/processed/mplads_cleaned.csv")
if not data_path.exists():
    data_path = Path("data/processed/mplads_cleaned.csv")

df = pd.read_csv(data_path, low_memory=False)
print(f"Loaded {len(df):,} records with {df.shape[1]} columns.")
df.info()

## 1. Core Financial Distributions
Examining sanctioned amounts, expenditure distributions, and spend variance across works.

In [ ]:
financial_cols = ['amount_sanctioned', 'amount_spent', 'cost_overrun_pct']
print("Descriptive Statistics for Financial Attributes:")
display(df[financial_cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(2))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(np.log10(df['amount_sanctioned'].clip(lower=1000)), kde=True, ax=axes[0], color="#2563EB", bins=40)
axes[0].set_title("Log-Transformed Sanctioned Amount (INR)")
axes[0].set_xlabel("Log10 (Amount Sanctioned)")

sns.boxplot(x=df['cost_overrun_pct'].clip(-20, 100), ax=axes[1], color="#EF4444")
axes[1].set_title("Cost Overrun Percentage Distribution (Clipped -20% to 100%)")
axes[1].set_xlabel("Cost Overrun (%)")
plt.tight_layout()
plt.show()

## 2. Sectoral and Geographic Work Distribution
Analyzing allocations across public work categories (Roads, Health, Education, Community Assets) and major Indian states.

In [ ]:
plt.figure(figsize=(14, 6))
top_states = df['state'].value_counts().head(15)
sns.barplot(x=top_states.values, y=top_states.index, palette="viridis")
plt.title("Top 15 States by Number of MPLADS Public Works")
plt.xlabel("Total Works Count")
plt.show()

plt.figure(figsize=(12, 5))
cat_counts = df['category'].value_counts()
sns.barplot(x=cat_counts.values, y=cat_counts.index, palette="crest")
plt.title("MPLADS Project Distribution Across Sectors")
plt.xlabel("Total Works Count")
plt.show()

## 3. Timeline Health & Implementation Lag
Evaluating project durations and days behind schedule relative to sanctioned milestones.

In [ ]:
if 'days_behind_schedule' in df.columns:
    print("Delay Metrics Overview:")
    print(f"Projects on schedule (delay == 0): {(df['days_behind_schedule'] == 0).sum():,} ({((df['days_behind_schedule'] == 0).mean()*100):.1f}%)")
    print(f"Delayed > 30 days: {(df['days_behind_schedule'] > 30).sum():,} ({((df['days_behind_schedule'] > 30).mean()*100):.1f}%)")
    print(f"Severely delayed > 180 days: {(df['days_behind_schedule'] > 180).sum():,} ({((df['days_behind_schedule'] > 180).mean()*100):.1f}%)")